# Модель на CoBaLD

In [1]:
pip install pyconll

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
import pyconll

In [7]:
class CustomCoNLLDataset(Dataset):
    def __init__(self, conllu_file, tokenizer, max_length=128, target_column=-1):
        self.data, self.labels = [], set()
        current_sentence, current_labels = [], []
        with open(conllu_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    if not line and current_sentence:
                        self.data.append((current_sentence.copy(), current_labels.copy()))
                        current_sentence, current_labels = [], []
                    continue
                parts = line.split('\t')
                if parts[0].isdigit() or '-' in parts[0]:
                    word = parts[1]
                    sem_class = parts[target_column] if len(parts) > 10 else 'O'
                    current_sentence.append(word)
                    current_labels.append(sem_class)
                    self.labels.add(sem_class)
            if current_sentence:
                self.data.append((current_sentence, current_labels))

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label2id = {l:i for i,l in enumerate(sorted(self.labels))}
        self.id2label = {i:l for l,i in self.label2id.items()}

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        text = ' '.join(tokens)
        encoding = self.tokenizer(text, truncation=True, padding='max_length',
                                  max_length=self.max_length, return_tensors='pt')
        bert_tokens = self.tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])
        token_labels = torch.ones(self.max_length, dtype=torch.long) * -100
        lbl_idx = 0
        for i, token in enumerate(bert_tokens):
            if token.startswith('##'):
                if i>0 and token_labels[i-1]!=-100:
                    token_labels[i] = token_labels[i-1]
            elif token in ['[CLS]','[SEP]','[PAD]']:
                continue
            elif lbl_idx < len(labels):
                token_labels[i] = self.label2id.get(labels[lbl_idx],0)
                lbl_idx +=1
        return {'input_ids':encoding['input_ids'].squeeze(),
                'attention_mask':encoding['attention_mask'].squeeze(),
                'labels':token_labels}

class SemanticModel(nn.Module):
    def __init__(self, bert_model, num_labels):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(seq_out)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
            return {'loss':loss, 'logits':logits}
        return {'logits':logits}

def train_semantic(conllu_file, model_name='bert-base-cased', device=None, epochs=3):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert = AutoModel.from_pretrained(model_name)
    ds = CustomCoNLLDataset(conllu_file, tokenizer)
    dl = DataLoader(ds, batch_size=16, shuffle=True)
    model = SemanticModel(bert, len(ds.label2id)).to(device)
    optim = AdamW(model.parameters(), lr=2e-5)
    model.train()
    for epoch in range(epochs):
        total=0
        for b in dl:
            optim.zero_grad()
            inp=b['input_ids'].to(device); att=b['attention_mask'].to(device); lbl=b['labels'].to(device)
            out=model(inp, att, lbl)
            out['loss'].backward(); optim.step()
            total+=out['loss'].item()
        print(f"Semantic Epoch {epoch+1}, Loss={total/len(dl):.4f}")
    return model, tokenizer, ds.id2label

class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, sem_model, id2label, max_length=128, device=None):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.sem_model = sem_model.to(device)
        self.id2label = id2label
        self.max_length = max_length
        self.device = device
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text=self.texts[idx]
        enc=self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        input_ids, att = enc['input_ids'].squeeze(), enc['attention_mask'].squeeze()
        with torch.no_grad():
            sem_out=self.sem_model(input_ids=input_ids.unsqueeze(0).to(self.device),
                                   attention_mask=att.unsqueeze(0).to(self.device))['logits']
        sem_feats = sem_out.squeeze().cpu()  # [seq_len, sem_labels]
        return {'input_ids':input_ids,'attention_mask':att,
                'sem_feats':sem_feats,'label':torch.tensor(self.labels[idx],dtype=torch.long)}

class SarcasmClassifier(nn.Module):
    def __init__(self, bert_model, sem_dim, num_classes):
        super().__init__()
        self.bert = bert_model
        self.sem_proj = nn.Linear(sem_dim, bert_model.config.hidden_size)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size*2, num_classes)
    def forward(self, input_ids, attention_mask, sem_feats, labels=None):
        bert_out=self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        sem_pooled = sem_feats.mean(dim=1)
        sem_proj = self.sem_proj(sem_pooled)
        joint = torch.cat([bert_out, sem_proj], dim=1)
        joint = self.dropout(joint)
        logits = self.classifier(joint)
        loss=None
        if labels is not None:
            loss_fct=nn.CrossEntropyLoss()
            loss=loss_fct(logits, labels)
            return {'loss':loss,'logits':logits}
        return logits

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Тренировка семантической модели
train_conllu = r'/content/train.conllu'
sem_model, tokenizer, id2label = train_semantic(train_conllu, device=device)
sem_model.eval()

Semantic Epoch 1, Loss=2.6116
Semantic Epoch 2, Loss=1.3290
Semantic Epoch 3, Loss=0.9583


SemanticModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [8]:
# Загрузка датасета сарказма
df = pd.read_csv(r'/content/dataset_all_data (2).csv')
ds = SarcasmDataset(df, tokenizer, sem_model, id2label, device=device)
dl = DataLoader(ds, batch_size=16, shuffle=True)

# Создание и тренировка классификатора сарказма
bert = sem_model.bert
sarcasm_model = SarcasmClassifier(bert, sem_dim=len(id2label), num_classes=2).to(device)
optimizer = AdamW(sarcasm_model.parameters(), lr=2e-5)

In [9]:
sarcasm_model.train()
for epoch in range(3):
    total_loss = 0
    for batch in dl:
        optimizer.zero_grad()
        out = sarcasm_model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['sem_feats'].to(device),
            batch['label'].to(device)
        )
        out['loss'].backward()
        optimizer.step()
        total_loss += out['loss'].item()
    print(f"Sarcasm Epoch {epoch+1}, Loss={total_loss/len(dl):.4f}")

# Сохранение модели
torch.save(sarcasm_model.state_dict(), 'sarcasm_model.pt')
print("Training complete.")

Sarcasm Epoch 1, Loss=0.4563
Sarcasm Epoch 2, Loss=0.4381
Sarcasm Epoch 3, Loss=0.4093
Training complete.


In [ ]:
import torch
import numpy as np
import pandas as pd

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Список текстов
texts = df['text'].tolist()

# Генерация усреднённых семантических векторов
sem_pooled_list = []
sem_model.to(device)
for text in texts:
    # Токенизация
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)

    # Предсказание логитов
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']

    # Усреднение по seq_len = вектор
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()
    sem_pooled_list.append(pooled)

df['sem_pooled'] = sem_pooled_list
df.to_pickle('enriched_sarcasm_pooled.pkl')
print(f"Сохранено {len(df)} записей в 'enriched_sarcasm_pooled.pkl' вот так")

Сохранено 9144 записей в 'enriched_sarcasm_pooled.pkl'


In [5]:
abc = pd.read_pickle(r'/content/enriched_sarcasm_pooled.pkl')

## Обучение

In [6]:
import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

class CustomBertClassifier(nn.Module):
    def __init__(self,
                 pretrained_model_name: str = 'bert-base-uncased',
                 num_labels: int = 2,
                 hidden_dim: int = 768,
                 dropout_prob: float = 0.1):
        super().__init__()
        # Базовая модель BERT без головы для маскированного языка
        self.bert = BertModel.from_pretrained(pretrained_model_name)

        # Кастомная голова
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self,
                input_ids: torch.LongTensor,
                attention_mask: torch.Tensor = None,
                token_type_ids: torch.Tensor = None,
                labels: torch.LongTensor = None):
        # Получаем выходы из берта
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        # Выбираем pooled_output
        pooled_output = outputs.pooler_output

        # Передаём через свою голову
        logits = self.classifier(pooled_output)

        # Если есть метки - считаем loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {
                'loss': loss,
                'logits': logits
            }
        return {'logits': logits}

In [7]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import BertTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from dataclasses import dataclass

combined_df = abc
combined_df['sarcasm'] = combined_df['sarcasm'].astype(int)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df['sarcasm'],
    random_state=42
)

# Токенизация
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

@dataclass
class NERFeatures:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    token_type_ids: torch.Tensor
    labels: torch.Tensor

class SarcasmDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаем модели
train_dataset = SarcasmDataset(train_df, tokenizer)
test_dataset  = SarcasmDataset(test_df, tokenizer)

# Загрузка модели
model = CustomBertClassifier(
    pretrained_model_name='bert-base-uncased',
    num_labels=2,
    hidden_dim=768,
    dropout_prob=0.1
)

# Метрики

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro')
    }

# Параметры
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

# Трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
results = trainer.evaluate()
print(results)
# Save the best model
trainer.save_model('./best_model')


Step,Training Loss
50,0.495600
100,0.472200
150,0.458700
200,0.434600
250,0.422300
300,0.426100
350,0.459900
400,0.431500
450,0.449500
500,0.462800


{'eval_loss': 0.43432796001434326, 'eval_accuracy': 0.8316019682886824, 'eval_f1': 0.5456413730803975, 'eval_runtime': 3.7381, 'eval_samples_per_second': 489.28, 'eval_steps_per_second': 7.758, 'epoch': 3.0}
